**[🏠 Course Home](../README.md)** | **📖 [Conceptual](../conceptual/06_quadratic_laplace_approximation.ipynb)** | **🤖 [AI-Native](06_quadratic_laplace_approximation.ipynb)** | **🐍 [Python](../python/06_quadratic_laplace_approximation.ipynb)** | **🔵 [R](../r/06_quadratic_laplace_approximation.ipynb)** | **⬅️ [Prev: Ch 05](05_frequentist_vs_grid_approximation.ipynb)** | **➡️ [Next: Ch 07](07_understanding_the_hessian_matrix.ipynb)**

*🧭 Part II: Numerical & Sampling Engines — Chapter 06 of 22*

---

# Chapter 6: The Laplace Shortcut — Quadratic (Laplace) Approximation
### *The Naive Gaussian Trap, Logit Transformations, and Curvature Audits*

> **🧭 Course Context & Why This Module Matters in the AI Era:**
> In Chapters 1–4, we explored exact conjugate updating, and in Chapter 5 we
> encountered the curse of dimensionality with grid approximation. The
> Quadratic (Laplace) Approximation is the first scalable numerical engine in
> Bayesian inference: it approximates an intractable continuous posterior
> distribution with a multivariate Gaussian centered at the Maximum A
> Posteriori (MAP) estimate, using the curvature (negative Hessian) to define
> parameter covariance. In modern AI workflows, Large Language Models (LLMs)
> routinely recommend Laplace approximations because they are computationally
> instantaneous. However, naive AI implementations almost universally fall into
> catastrophic boundary traps—approximating naturally bounded parameters (such
> as failure probabilities $p \in [0, 1]$ or standard deviations $\sigma > 0$)
> with unbounded symmetric bells that spill mass into impossible physical
> regimes. This notebook establishes the epistemic contract for AI-driven
> quadratic approximations.

## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

When software engineers attempt to evaluate rare-event failure rates or service
reliability using an LLM, they typically issue an unconstrained natural
language prompt.

### The Naive Prompt
```text
I observed 1 failure across 100 canary test executions. Fit a quick Bayesian
Laplace quadratic approximation to estimate the failure rate p and give me a 95%
credible interval.
```

### The Plausible-Sounding AI Hallucination
An unconstrained LLM generates the following confident but mathematically
absurd response:
> *"We find the MAP estimate for the binomial likelihood with $k=1, N=100$ and
> a uniform prior $\text{Beta}(1, 1)$. The posterior mode is $\hat{p} = 1/100 =
> 0.01$. The observed Fisher information (negative second derivative of the log
> posterior) is $I(\hat{p}) = N / (\hat{p}(1 - \hat{p})) = 100 / (0.01 \times
> 0.99) \approx 10,101.01$. The posterior variance is the inverse of curvature:
> $\sigma_p^2 = 1 / I(\hat{p}) \approx 0.000099$, so $\sigma_p \approx
> 0.00995$. Using a Gaussian Laplace approximation $\mathcal{N}(0.01,
> 0.00995^2)$, the 95% credible interval is:
> $$0.01 \pm 1.96 \times 0.00995 = [-0.0095, 0.0295]$$
> Conclusion: The canary failure rate is 95% likely to be between -0.95% and
> 2.95%."*

```
                 THE NAIVE AI BOUNDARY DISASTER
                 
    Impossible Negative Mass        Valid Physical Range
    (Probability < 0)               (Probability >= 0)
    <-------------------------> | <------------------------------------>
                                |
             * * * * *          |
          *             *       |
        *                 *     |   Mode = +0.01
       *                   *    |     |
      *                     *   |     v
     *                       *  |   * * *
    *                         * | *       *
   *                           *|           *
  ------------------------------+---------------------------------------
 -0.01                        0.00         +0.01       +0.02       +0.03
                                ^
                                |
            Physical Boundary: Probability Cannot Be Negative!
```

---

### Forensic Post-Mortem: Why the AI Failed
The LLM produced three severe epistemic errors that would invalidate downstream
automated safety decisions:

1. **Physical Support Violation ($p < 0$):**
   Probabilities are mathematically and physically confined to the compact
   interval $[0, 1]$. By placing an unconstrained Gaussian directly on $p$:
   $$\Phi\left(\frac{0 - 0.01}{0.00995}\right) = \Phi(-1.005) \approx 15.74\%$$
   Approximately 15.74% of the posterior mass is assigned to impossible
   negative failure rates!
2. **Severe Skewness Failure (Symmetry Delusion):**
   The exact conjugate posterior is $\text{Beta}(k+1, N-k+1) = \text{Beta}(2,
   100)$. The skewness of a Beta distribution is:
   $$\gamma_1 = \frac{2(\beta - \alpha)\sqrt{\alpha + \beta + 1}}{(\alpha + \beta + 2)\sqrt{\alpha\beta}} = \frac{2(100 - 2)\sqrt{103}}{104\sqrt{200}} \approx \frac{196 \times 10.1488}{104 \times 14.142} \approx 1.353$$
   A distribution with skewness $+1.35$ has a heavy right tail. A Gaussian
   quadratic approximation has skewness identically equal to $0$. The naive AI
   forced a severely skewed distribution into a symmetric mold.
3. **Absence of Coordinate Transformation:**
   A calibrated probabilistic engineer never fits a quadratic approximation on
   bounded coordinates. The parameter must first be mapped to an unconstrained
   domain $\mathbb{R}$ via the log-odds (logit) transform:
   $$\phi = \text{logit}(p) = \log\left(\frac{p}{1 - p}\right) \in (-\infty, +\infty)$$
   Only on unconstrained space $\phi$ does a parabolic Taylor expansion
   preserve physical validity upon back-transformation via the logistic
   sigmoid $p = \sigma(\phi) = \frac{1}{1 + e^{-\phi}}$.

## Part 2: Prompting as Probabilistic Programming — Declarative Specification

To prevent boundary violations and unphysical approximations, we instruct the
AI agent using a structured declarative specification block that enforces
unconstrained coordinate reparameterization and Jacobian corrections.

```yaml
# ==============================================================================
# PROBABILISTIC SPECIFICATION: QUADRATIC (LAPLACE) APPROXIMATION CONTRACT
# ==============================================================================
module: "ai/06_quadratic_laplace_approximation"
target_parameter:
  name: "failure_rate_p"
  physical_domain: "[0.0, 1.0]"
  transform_to_unconstrained: "phi = log(p / (1.0 - p))"
  inverse_transform: "p = 1.0 / (1.0 + exp(-phi))"
  jacobian_adjustment_log_det: "log(p) + log(1.0 - p)"

observables:
  observed_successes: 99
  observed_failures: 1
  total_trials: 100

prior_model:
  parameter_space: "unconstrained_phi"
  # Weakly informative prior on log-odds
  distribution: "Normal(loc=0.0, scale=2.5)"
  implied_probability_median: 0.50
  implied_probability_95_interval: "[0.007, 0.993]"

likelihood_model:
  family: "Binomial(n=total_trials, p=inv_logit(phi))"
  log_likelihood_formula: "k * phi - n * log(1.0 + exp(phi))"

computational_invariants:
  optimization_target: "negative_log_joint(phi)"
  optimizer: "scipy.optimize.minimize(method='BFGS')"
  curvature_engine: "observed_information = -hessian(phi_map)"
  admissibility_conditions:
    - "hessian_eigenvalues > 0.0"  # Must be strictly positive definite
    - "condition_number < 1e4"
    - "skewness_absolute_threshold < 0.3"
    - "boundary_leakage_integral == 0.0"

decision_loss_contract:
  credible_interval_type: "highest_density_interval_or_quantile"
  nominal_coverage: 0.95
  report_format: "median_and_interval"
  unconstrained_to_constrained_mapping: "sample_draws_then_transform"
```

## Part 3: The 4 Mandatory Agentic Verification Gates

Before accepting any Laplace approximation output from an AI agent, the
following four verification gates must execute deterministically:

### Gate 1: Prior Predictive Check Gate
* **Invariant:** Prior draws must respect physical support boundaries and not
  concentrate excessive mass at extremal boundaries.
* **Verification Logic:**
  Draw $10,000$ samples from the prior on $\phi \sim \mathcal{N}(0, 2.5^2)$.
  Transform each draw to $p = \sigma(\phi)$. Verify that $p \in (0, 1)$ strictly
  for $100\%$ of draws, and that the prior predictive distribution for canary
  failures produces realistic counts rather than polar spikes at $0$ or $N$.

### Gate 2: Fake-Data Recovery / Simulation-Based Calibration (SBC) Gate
* **Invariant:** If data are simulated from known ground truth $p^* \in \{0.005,
  0.02, 0.10, 0.50\}$, Laplace in logit space must recover $p^*$ within its
  90% credible interval in approximately 90% of repeated trials.
* **Verification Logic:**
  Run $M=500$ synthetic replications. For each replication, fit the Laplace
  approximation in $\phi$-space. Transform the $90\%$ interval back to $[0, 1]$.
  Assert empirical coverage falls within $[87\%, 93\%]$. Flag if boundary
  proximity ($p^* < 0.01$) degrades coverage below nominal tolerance.

### Gate 3: Numerical & Analytical Invariants Gate
* **Invariant 3A (Positive Curvature):**
  The observed information matrix (negative Hessian at MAP mode) must satisfy
  $\lambda_{\min} > 0$. If $\lambda_{\min} \le 0$, the optimizer halted at a
  saddle point or flat plain; abort immediately.
* **Invariant 3B (Zero Boundary Leakage):**
  $$\int_{-\infty}^0 P_{\text{approx}}(p) \, dp = 0.000000$$
  By construction, transforming samples from unconstrained $\phi \in
  (-\infty, \infty)$ via the logistic sigmoid guarantees that zero probability
  mass leaks into negative territory.
* **Invariant 3C (Third Derivative Skewness Audit):**
  Evaluate the third derivative of the log-posterior at the mode:
  $$\Delta_{\text{skew}} = \left| \frac{d^3 \log p(\phi \mid \mathcal{D})}{d\phi^3} \cdot \sigma_\phi^3 \right|$$
  If $\Delta_{\text{skew}} > 0.5$, warn that the quadratic approximation is
  insufficiently capturing posterior asymmetry and escalate to MCMC sampling.

### Gate 4: Posterior Predictive Retrodiction Gate
* **Invariant:** Observed count $k_{\text{obs}} = 1$ must fall within the 95%
  central interval of the posterior predictive distribution:
  $$y_{\text{rep}} \sim \text{Binomial}(N=100, p_{\text{draw}})$$
* **Verification Logic:**
  Generate $10,000$ draws of $p_{\text{draw}} = \sigma(\phi_{\text{draw}})$
  where $\phi_{\text{draw}} \sim \mathcal{N}(\phi_{\text{MAP}}, \sigma_\phi^2)$.
  Simulate $y_{\text{rep}} \sim \text{Binomial}(100, p_{\text{draw}})$. Verify
  that $P(y_{\text{rep}} = 1) > 0.05$ and $k=1$ is in the support.

## Part 4: Worked Agent Execution Transcript & Telemetry

When an epistemic AI agent executes the declarative specification on $k=1,
N=100$, it produces the following verified execution transcript:

```text
[STEP 1: COORDINATE INITIALIZATION]
  Target Parameter: p in [0, 1]
  Reparameterization: phi = logit(p) in (-inf, +inf)
  Prior: phi ~ Normal(0.0, 2.5)
  Observed Data: k = 1, N = 100

[STEP 2: OPTIMIZATION IN UNCONSTRAINED SPACE]
  Negative Log-Posterior:
    -log p(phi | D) = -(k * phi - N * log(1 + exp(phi)) - 0.5 * (phi / 2.5)^2)
  Initial Guess: phi_0 = 0.0
  Optimization Algorithm: BFGS (Scipy)
  Optimization Status: Converged successfully in 7 iterations
  Optimal Unconstrained Mode (MAP): phi_hat = -4.5103
  Implied Probability MAP: p_hat = 1 / (1 + exp(4.5103)) = 0.01088 (1.09%)

[STEP 3: HESSIAN EVALUATION & INVERSION]
  Second Derivative:
    d^2(-log p)/dphi^2 = N * exp(phi) / (1 + exp(phi))^2 + (1 / 2.5^2)
  At phi_hat = -4.5103:
    Curvature H = 100 * (0.01088 * 0.98912) + 0.16 = 1.0763 + 0.16 = 1.2363
  Posterior Variance: sigma_phi^2 = 1 / H = 1 / 1.2363 = 0.80887
  Posterior StdDev in Logit Space: sigma_phi = sqrt(0.80887) = 0.89937

[STEP 4: UNCONSTRAINED POSTERIOR INFERENCE]
  Unconstrained 95% Credible Interval (phi-space):
    phi_lower = -4.5103 - 1.95996 * 0.89937 = -6.2730
    phi_upper = -4.5103 + 1.95996 * 0.89937 = -2.7476

[STEP 5: NON-LINEAR BACK-TRANSFORMATION TO PROBABILITY SPACE]
  p_lower = 1 / (1 + exp(6.2730)) = 0.00188  (0.19%)
  p_median = 1 / (1 + exp(4.5103)) = 0.01088 (1.09%)
  p_upper = 1 / (1 + exp(2.7476)) = 0.06014  (6.01%)
  Boundary Leakage P(p < 0): 0.00000000% (STRICTLY IMPOSSIBLE)
```

### ASCII Comparative Posterior Distribution
```
 Probability Density Comparison: True Beta vs. Naive vs. Calibrated Laplace
 
 Density
   |
 40+         True Posterior: Beta(2, 101)  [Exact Conjugate]
   |         ..................
 30+        :                  :
   |       :                    :           Naive Gaussian (Unbounded)
 20+      :                      :        x x x x x x x x x x x x x
   |     :                        :      x                         x
 10+    :                          :    x                           x
   |   :                            :  x                             x
  0+--:------------------------------xx-------------------------------+---> p
   -0.01     0.00     +0.01    +0.02    +0.03    +0.04    +0.05    +0.06
               ^
               | Physical Boundary p = 0
               |
  [Naive Gaussian Spills 15.7% Mass into p < 0!]
  [Calibrated Transformed Laplace Stays Strictly in [0, 1] with Positive Skew!]
```

### Decision Telemetry Table
| Method | Coordinate Space | Point Estimate (Mode/Median) | 95% Credible Interval | P(p < 0) Leakage | Skewness Captured |
|---|---|---|---|---|---|
| **Naive AI Prompt** | Raw $p \in [0, 1]$ | $\hat{p} = 0.0100$ | $[-0.0095, 0.0295]$ | **15.74%** (Fatal) | $0.00$ (Rejected) |
| **Exact Conjugate** | Probability $p$ | $\text{Mode} = 0.0099$ | $[0.0024, 0.0543]$ | **0.00%** (Exact) | $+1.35$ (Heavy Tail) |
| **Calibrated Laplace** | Log-odds $\phi$ | $\text{Median} = 0.0109$ | $[0.0019, 0.0601]$ | **0.00%** (Guaranteed) | $+1.28$ (Preserved) |

## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Dimension | Naive AI Prompt | Calibrated Epistemic Specification |
|---|---|---|
| **Parameter Space** | Unconstrained Gaussian fitted directly to bounded parameter | Mapped to $\mathbb{R}$ via non-linear transform (logit, log) |
| **Physical Support** | Leaks massive probability mass into negative/unphysical values | Guaranteed 100% adherence to physical boundary $[0, 1]$ |
| **Skewness Handling** | Forces symmetric bell curve; severely underestimates tail risk | Preserves asymmetric tail under inverse non-linear mapping |
| **Observed Information** | Treats $I(\theta)$ naively; ignores Jacobian adjustments | Rigorously accounts for parameter transformation curvature |
| **Verification Gates** | None; accepts whatever numbers the optimizer spits out | 4 gates: Prior predictive, SBC recovery, Invariants, PPC |
| **Production Decision** | "Canary failure rate could be -0.5% (Pass)" | "Canary failure 95% CI is [0.19%, 6.01%]; triggers caution" |

## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

### 🟢 Tier 1: Spot the AI Hallucination
An engineer asks an AI assistant to calculate the uncertainty of a variance
component in a server response-time model:
> *"I fit a normal distribution to $N=25$ latency samples. The MAP estimate
> for the standard deviation is $\hat{\sigma} = 2.3$ ms. The second derivative
> of the negative log-likelihood at the mode is $H = 2N / \hat{\sigma}^2 = 50 /
> 5.29 \approx 9.45$. The Laplace variance is $1/9.45 = 0.1058$, so the standard
> error is $\text{SE} = \sqrt{0.1058} = 0.325$. The 99% credible interval for
> $\sigma$ is $2.3 \pm 2.576 \times 0.325 = [1.46, 3.14]$ ms."*

> Later, the engineer runs the same code on a smaller dataset of $N=4$ where
> $\hat{\sigma} = 0.8$ ms, $H = 8 / 0.64 = 12.5$, $\text{SE} = 1/\sqrt{12.5}
> \approx 0.283$, and the AI reports a 99% credible interval of:
> $$0.8 \pm 2.576 \times 0.283 = [-0.07, 1.53] \text{ ms}$$

What is the fundamental statistical and architectural bug in the AI's logic?

<details>
<summary><b>Click here to view the solution...</b></summary>

**Forensic Audit:**
The AI fitted a Laplace Gaussian approximation directly in the scale coordinate
$\sigma \in (0, \infty)$.
1. **Support Violation:** Standard deviation is strictly non-negative ($\sigma
   > 0$). For small sample size $N=4$, the symmetric Gaussian distribution
   spreads across the boundary, resulting in a negative lower bound $[-0.07$ ms.
2. **Missing Transformation:** Scale parameters must be mapped to unconstrained
   real space via the log transform $\psi = \log \sigma \in (-\infty, \infty)$.
   In $\psi$-space, the curvature yields a Gaussian distribution on $\psi$,
   which when exponentiated $\sigma = \exp(\psi)$ produces a strictly positive,
   right-skewed Log-Normal distribution with zero probability below $0$.
</details>

---

### 🟡 Tier 2: Refactor the Flawed Specification
Below is an uncalibrated agent prompt designed to estimate the rate parameter
$\lambda$ of an exponential service time distribution:

```text
You are a Bayesian assistant. Fit a Laplace approximation to estimate the rate
parameter lambda > 0 from 8 service completion times. Calculate the second
derivative of the negative log-likelihood with respect to lambda, invert it to
get the variance, and output the normal 95% credible interval [lambda_hat -
1.96*SE, lambda_hat + 1.96*SE].
```

Rewrite this prompt into a calibrated declarative specification contract that
prevents boundary leakage and validates curvature.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Calibrated Declarative Specification:**
```yaml
module: "rate_estimation_contract"
parameter:
  name: "rate_lambda"
  constraint: "strictly_positive (lambda > 0)"
  unconstrained_parameter: "log_lambda = log(lambda)"
  inverse_transform: "lambda = exp(log_lambda)"
  jacobian_adjustment: "+log_lambda"

likelihood:
  family: "Exponential(rate=exp(log_lambda))"
  data: "service_times"

invariants:
  optimization: "BFGS on negative log posterior in log_lambda space"
  curvature_check:
    - "hessian > 0"
    - "skewness_third_derivative < 0.3"
  interval_construction:
    method: "Compute 95% interval in log_lambda space"
    formula: "[mu_log - 1.96 * se, mu_log + 1.96 * se]"
    projection: "Exponentiate interval bounds to get [exp(lower), exp(upper)]"
    assert_boundary: "assert lower_bound > 0.0"
```
</details>

---

### 🔴 Tier 3: Architect an Autonomous Agent Contract
Architect an autonomous agent verification contract in pseudo-code that
receives an arbitrary Bayesian model and dataset, attempts a Laplace
approximation, automatically evaluates the third derivative (skewness), and
executes an automated decision: accept Laplace approximation OR dynamically
escalate to Hamiltonian Monte Carlo (HMC).

<details>
<summary><b>Click here to view the solution...</b></summary>

**Autonomous Contract Architecture:**
```python
def autonomous_laplace_gatekeeper(log_post_func, theta_init, data):
  # 1. Transform parameters to unconstrained coordinates
  unconstrained_target, inv_map = enforce_unconstrained_support(log_post_func)

  # 2. Find MAP mode via numerical optimization
  opt_result = minimize_loss(
      lambda th: -unconstrained_target(th, data), theta_init
  )
  if not opt_result.converged:
    return {"status": "ESCALATE_TO_MCMC", "reason": "Optimization failed"}

  theta_map = opt_result.x
  H = compute_hessian(
      lambda th: -unconstrained_target(th, data), theta_map
  )

  # 3. Check positive definiteness
  eigenvalues = np.linalg.eigvalsh(H)
  if np.any(eigenvalues <= 0):
    return {
        "status": "ESCALATE_TO_MCMC",
        "reason": f"Non-convex curvature detected: min_eig={eigenvalues.min()}",
    }

  # 4. Compute third derivative (Skewness / Non-parabolic asymmetry)
  third_deriv = compute_third_derivative(
      lambda th: -unconstrained_target(th, data), theta_map
  )
  sigma_th = np.sqrt(np.diag(np.linalg.inv(H)))
  skewness_metric = np.max(np.abs(third_deriv * (sigma_th**3)))

  if skewness_metric > 0.25:
    return {
        "status": "ESCALATE_TO_MCMC",
        "reason": (
            f"Posterior severely non-quadratic (skewness metric"
            f" {skewness_metric:.3f} > 0.25)"
        ),
        "laplace_samples": None,
    }

  # 5. Passed all gates: Generate calibrated samples and project back
  vcov = np.linalg.inv(H)
  raw_samples = sample_multivariate_normal(theta_map, vcov, size=10000)
  projected_samples = inv_map(raw_samples)

  return {
      "status": "ACCEPTED_LAPLACE",
      "samples": projected_samples,
      "hdi_95": compute_hdi(projected_samples, 0.95),
  }
```
</details>

## Key Takeaways & Course Navigation

### 🎯 Key Takeaways:
1. **The Quadratic Illusion:** The Laplace approximation assumes the
   log-posterior surface is a perfect parabola. In finite samples near physical
   boundaries, this assumption violently breaks down.
2. **Support Preservation via Unconstrained Mapping:** Never apply a Laplace
   Gaussian approximation directly in constrained parameter space. Always
   transform bounded parameters ($p \in [0, 1] \to \text{logit}(p)$, $\sigma > 0
   \to \log \sigma$) before optimization.
3. **Curvature is Information:** The negative Hessian at the MAP estimate is the
   observed Fisher information. Inverting this curvature yields the covariance
   matrix of the Gaussian approximation.
4. **Autonomous Escalation:** An epistemic AI pipeline must compute the third
   derivative (skewness). When curvature changes rapidly across the mode, the
   system must reject Laplace and escalate to MCMC.

---

**[🏠 Course Home](../README.md)** | **📖 [Conceptual](../conceptual/06_quadratic_laplace_approximation.ipynb)** | **🤖 [AI-Native](06_quadratic_laplace_approximation.ipynb)** | **🐍 [Python](../python/06_quadratic_laplace_approximation.ipynb)** | **🔵 [R](../r/06_quadratic_laplace_approximation.ipynb)** | **⬅️ [Prev: Ch 05](05_frequentist_vs_grid_approximation.ipynb)** | **➡️ [Next: Ch 07](07_understanding_the_hessian_matrix.ipynb)**